# StormEngine — DPC/Virtual Station Data Quality & Analysis

Complete four-step quality control and analysis pipeline for DPC/Virtual station data.

| Step | Content |
|---|---|
| **1** | Sensor inventory & data dictionary |
| **2** | Real-time data quality (range check, flat-line, gap detection) |
| **3** | Cross-match DPC vs ERA5 — spatial bias profile |
| **4** | Radar QC — coverage, artefacts, temporal alignment |

## Input file structure
| Column | Description |
|---|---|
| `station_id` | Unique station identifier |
| `sensor_code` | Variable code (TARIA2M, PRESS, VV, PREC) |
| `quantity` | Physical variable name |
| `unit` | Measurement unit |
| `dt` | Observation timestamp |
| `value` | Measured value |
| `lat`, `lon` | Station coordinates |
| `quota` | Station elevation (m) |
| `gestore` | Data provider |
| `provincia` | Administrative region |

## 0. Imports and Configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import glob, os, json, warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 110, 'savefig.dpi': 200, 'font.size': 11,
    'axes.titlesize': 13, 'axes.titleweight': 'bold',
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.25, 'legend.frameon': False,
})

# ── Input files ─────────────────────────────────────────────────────────────
# List all DPC/Virtual CSV files — add more files as they are collected
DPC_FILES = sorted(glob.glob('*.csv'))  # all CSVs in same folder
# Filter to DPC/Virtual files only
DPC_FILES = [f for f in DPC_FILES if any(
    k in f.lower() for k in ['dpc','fvg','ven','emi','mar','abr','pug','virtual','latest']
)]
print('DPC files found:', DPC_FILES)

# ERA5 files for cross-match (Step 3)
ERA5_FILES = {
    'November' : {'path': 'final_2024_11_msl.csv', 'year': 2024, 'month': 11},
    'December' : {'path': 'final_2024_12_msl.csv', 'year': 2024, 'month': 12},
    'January'  : {'path': 'final_2024_1_msl.csv',  'year': 2025, 'month':  1},
    'February' : {'path': 'final_2024_2_msl.csv',  'year': 2025, 'month':  2},
}

# Radar GeoTIFF folder (Step 4)
RADAR_DIR = 'radar/'   # update to your radar folder

# ── Domain ───────────────────────────────────────────────────────────────────
LAT_MIN, LAT_MAX = 39.0, 46.5
LON_MIN, LON_MAX = 12.0, 20.0
PA_TO_HPA = 1.0 / 100.0

# ── Physical plausibility bounds per sensor_code ──────────────────────────────
PHYS_BOUNDS = {
    'TARIA2M' : (-20.0,  50.0),   # °C
    'PRESS'   : (950.0, 1060.0),  # hPa
    'VV'      : (  0.0,  60.0),   # m/s
    'PREC'    : (  0.0, 200.0),   # mm/h
    'U10'     : (-60.0,  60.0),   # m/s
    'V10'     : (-60.0,  60.0),   # m/s
}

SENSOR_COLORS = {
    'TARIA2M': '#e6550d',
    'PRESS'  : '#3182bd',
    'VV'     : '#31a354',
    'PREC'   : '#756bb1',
}

# Flat-line detection threshold
FLATLINE_HOURS = 3    # flag if same value for >= N consecutive hours
# Gap detection threshold
GAP_HOURS = 2         # flag gaps > N hours between consecutive obs

print('Configuration loaded.')

---
# STEP 1 — Sensor Inventory & Data Dictionary

Documents all active sensors across stations: which variables are measured,
units, update frequency, elevation, and data provider.
This constitutes the formal data dictionary for the report.

## 1.1 Load All DPC CSV Files

In [ ]:
def load_dpc(files):
    """Load and concatenate all DPC/Virtual CSV files."""
    if not files:
        raise FileNotFoundError('No DPC CSV files found. Check DPC_FILES.')
    dfs = []
    for f in files:
        try:
            df = pd.read_csv(f)
            df['source_file'] = os.path.basename(f)
            dfs.append(df)
            print(f'  Loaded: {f}  ({len(df)} rows)')
        except Exception as e:
            print(f'  Warning: {f}: {e}')
    df = pd.concat(dfs, ignore_index=True)
    df['dt'] = pd.to_datetime(df['dt'])
    if 'aggiornamento' in df.columns:
        df['aggiornamento'] = pd.to_datetime(df['aggiornamento'])
    return df

dpc = load_dpc(DPC_FILES)
print(f'\nTotal rows       : {len(dpc)}')
print(f'Unique stations  : {dpc["station_id"].nunique()}')
print(f'Sensor codes     : {sorted(dpc["sensor_code"].unique())}')
print(f'Gestori          : {sorted(dpc["gestore"].unique())}')
print(f'Timestamp range  : {dpc["dt"].min()} → {dpc["dt"].max()}')

## 1.2 Data Dictionary — Sensor Inventory Table

In [ ]:
# Build sensor inventory: one row per (station, sensor_code)
inventory = dpc.groupby(['station_id','sensor_code','quantity','unit',
                         'lat','lon','quota','gestore','provincia']).agg(
    n_obs       = ('value', 'count'),
    first_obs   = ('dt', 'min'),
    last_obs    = ('dt', 'max'),
    value_min   = ('value', 'min'),
    value_max   = ('value', 'max'),
    value_mean  = ('value', 'mean'),
).reset_index().round(3)

# Infer update frequency from timestamp differences per station
freq_map = {}
for sid, grp in dpc.groupby('station_id'):
    g = grp.sort_values('dt')
    diffs = g['dt'].diff().dt.total_seconds() / 60  # minutes
    diffs = diffs.dropna()
    freq_map[sid] = f'{diffs.median():.0f} min' if len(diffs) > 0 else 'single snapshot'
inventory['update_freq'] = inventory['station_id'].map(freq_map)

print(f'Data dictionary: {len(inventory)} sensor-station pairs')
print(f'Active sensors by type:')
print(inventory.groupby('sensor_code')['station_id'].count().rename('n_stations').to_string())
print()
print('Sample rows:')
inventory.head(8)

## 1.3 Sensor Availability Heatmap

In [ ]:
# Which sensors are available at each station?
avail = dpc.groupby(['station_id','sensor_code']).size().unstack(fill_value=0)
avail_bool = (avail > 0).astype(int)

fig, ax = plt.subplots(figsize=(max(8, len(avail_bool.columns)*2+2),
                                 max(6, len(avail_bool)*0.25+2)))
im = ax.imshow(avail_bool.values, cmap='Blues', aspect='auto', vmin=0, vmax=1)
ax.set_xticks(range(len(avail_bool.columns)))
ax.set_xticklabels(avail_bool.columns, fontsize=10)
ax.set_yticks(range(len(avail_bool.index)))
ax.set_yticklabels([s[:20] for s in avail_bool.index], fontsize=6)
for i in range(len(avail_bool.index)):
    for j in range(len(avail_bool.columns)):
        ax.text(j, i, '✓' if avail_bool.values[i,j] else '✗',
                ha='center', va='center', fontsize=7,
                color='white' if avail_bool.values[i,j] else '#ccc')
plt.colorbar(im, ax=ax, label='Available')
ax.set_title('Step 1 — Sensor Availability per Station')
plt.tight_layout()
plt.savefig('s1_sensor_availability.png', bbox_inches='tight')
plt.show()

## 1.4 Station Map

In [ ]:
stations = dpc.drop_duplicates('station_id')[['station_id','lat','lon','quota','gestore','provincia']]
fig, ax = plt.subplots(figsize=(11, 8))
gestori = stations['gestore'].unique()
cmap_g = plt.cm.get_cmap('tab10', len(gestori))
for idx, g in enumerate(gestori):
    sub = stations[stations['gestore']==g]
    ax.scatter(sub['lon'], sub['lat'], s=60, color=cmap_g(idx),
               edgecolors='black', linewidths=0.4, zorder=5, label=g)
for _, row in stations.iterrows():
    ax.annotate(row['station_id'][:12],
                (row['lon'], row['lat']), fontsize=5,
                xytext=(3,3), textcoords='offset points', color='black')
ax.set(xlim=(LON_MIN-0.3, LON_MAX+0.3), ylim=(LAT_MIN-0.3, LAT_MAX+0.3),
       xlabel='Longitude (°E)', ylabel='Latitude (°N)',
       title=f'Step 1 — Station Map  ({len(stations)} stations)')
ax.legend(loc='lower right', fontsize=8, ncol=2)
plt.tight_layout()
plt.savefig('s1_station_map.png', bbox_inches='tight')
plt.show()

## 1.5 Export Data Dictionary

In [ ]:
inventory.to_csv('dpc_data_dictionary.csv', index=False)
print('Saved -> dpc_data_dictionary.csv')
print(f'Columns: {list(inventory.columns)}')

---
# STEP 2 — Real-Time Data Quality

Three QC checks applied to DPC data since no systematic QC exists:
- **Range check** — physical plausibility bounds per variable
- **Flat-line detection** — sensor stuck on same value for N consecutive hours
- **Gap detection** — station offline for more than N hours

## 2.1 Physical Range Check

In [ ]:
print('='*55)
print('  2.1 PHYSICAL RANGE CHECK')
print('='*55)

range_flags = []
for code, (lo, hi) in PHYS_BOUNDS.items():
    sub = dpc[dpc['sensor_code']==code]
    if len(sub) == 0: continue
    bad = sub[(sub['value']<lo)|(sub['value']>hi)]
    pct = 100*len(bad)/len(sub) if len(sub)>0 else 0
    print(f'  {code:8s}: {len(bad):5d} out of {len(sub):6d} ({pct:.2f}%)  bounds=[{lo},{hi}]')
    range_flags.append({'sensor_code':code, 'n_total':len(sub),
                        'n_bad':len(bad), 'pct_bad':pct})
    if len(bad)>0:
        print(f'  Sample bad rows:')
        print(bad[['station_id','dt','value']].head(3).to_string(index=False))

range_df = pd.DataFrame(range_flags)

# Mark bad rows
dpc['range_flag'] = False
for code, (lo, hi) in PHYS_BOUNDS.items():
    mask = (dpc['sensor_code']==code) & ((dpc['value']<lo)|(dpc['value']>hi))
    dpc.loc[mask, 'range_flag'] = True

# Visualise value distributions with bounds
codes = [c for c in PHYS_BOUNDS if c in dpc['sensor_code'].values]
fig, axes = plt.subplots(1, len(codes), figsize=(5*len(codes), 4))
if len(codes)==1: axes=[axes]
for ax, c in zip(axes, codes):
    sub = dpc[dpc['sensor_code']==c]['value']
    ax.hist(sub, bins=40, color=SENSOR_COLORS.get(c,'steelblue'),
            alpha=0.7, edgecolor='white', linewidth=0.3)
    lo, hi = PHYS_BOUNDS[c]
    ax.axvline(lo, color='red', linestyle=':', linewidth=1.2)
    ax.axvline(hi, color='red', linestyle=':', linewidth=1.2,
               label=f'Bounds [{lo},{hi}]')
    unit = dpc[dpc['sensor_code']==c]['unit'].iloc[0]
    ax.set(xlabel=f'{c} ({unit})', ylabel='Count', title=f'{c} distribution')
    ax.legend(fontsize=8)
fig.suptitle('Step 2 — Physical Range Check', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('s2_range_check.png', bbox_inches='tight')
plt.show()

## 2.2 Flat-line Detection

In [ ]:
print('='*55)
print('  2.2 FLAT-LINE DETECTION')
print('='*55)

def detect_flatlines(df, n_hours=3):
    """Flag stations/sensors where value is constant for >= n_hours consecutive obs."""
    results = []
    for (sid, sc), grp in df.groupby(['station_id','sensor_code']):
        grp = grp.sort_values('dt').copy()
        if len(grp) < n_hours: continue
        # Count consecutive runs of same value
        diff = grp['value'].diff().ne(0).cumsum()
        run_len = diff.map(diff.value_counts())
        max_run = run_len.max()
        if max_run >= n_hours:
            stuck_val = grp.loc[run_len.idxmax(), 'value']
            results.append({'station_id':sid,'sensor_code':sc,
                            'max_run_hours':max_run,'stuck_value':stuck_val})
    return pd.DataFrame(results)

flatlines = detect_flatlines(dpc, n_hours=FLATLINE_HOURS)
print(f'  Flat-line threshold : >= {FLATLINE_HOURS} consecutive identical values')
print(f'  Flagged pairs       : {len(flatlines)}')

if len(flatlines)>0:
    print(flatlines.sort_values('max_run_hours',ascending=False).head(15).to_string(index=False))
else:
    print('  No flat-line issues detected — all sensors show variation.')

dpc['flatline_flag'] = dpc.apply(
    lambda r: ((flatlines['station_id']==r['station_id']) &
               (flatlines['sensor_code']==r['sensor_code'])).any()
    if len(flatlines)>0 else False, axis=1
)

# Visualise if any found
if len(flatlines)>0:
    worst = flatlines.sort_values('max_run_hours',ascending=False).iloc[0]
    sub = dpc[(dpc['station_id']==worst['station_id']) &
              (dpc['sensor_code']==worst['sensor_code'])].sort_values('dt')
    fig, ax = plt.subplots(figsize=(12,4))
    ax.plot(sub['dt'], sub['value'], 'o-', markersize=3, linewidth=1)
    ax.set(xlabel='Time', ylabel=f"{worst['sensor_code']}",
           title=f"Step 2 — Flat-line example: {worst['station_id']} / {worst['sensor_code']}\n"
                 f"max run = {worst['max_run_hours']} h  stuck value = {worst['stuck_value']}")
    plt.tight_layout()
    plt.savefig('s2_flatline.png', bbox_inches='tight')
    plt.show()

## 2.3 Gap Detection

In [ ]:
print('='*55)
print('  2.3 GAP DETECTION')
print('='*55)

def detect_gaps(df, gap_hours=2):
    """Flag temporal gaps > gap_hours between consecutive observations per station."""
    results = []
    for sid, grp in df.groupby('station_id'):
        grp = grp.sort_values('dt').drop_duplicates('dt')
        diffs = grp['dt'].diff().dt.total_seconds()/3600
        big_gaps = diffs[diffs > gap_hours]
        if len(big_gaps)>0:
            for idx in big_gaps.index:
                results.append({'station_id':sid,
                                'gap_start':grp.loc[idx,'dt'] - pd.Timedelta(hours=diffs[idx]),
                                'gap_end':grp.loc[idx,'dt'],
                                'gap_hours':diffs[idx]})
    return pd.DataFrame(results)

gaps = detect_gaps(dpc, gap_hours=GAP_HOURS)
print(f'  Gap threshold  : > {GAP_HOURS} hours')
print(f'  Gaps detected  : {len(gaps)}')

if len(gaps)>0:
    print(f'  Stations with gaps: {gaps["station_id"].nunique()}')
    print(gaps.sort_values('gap_hours',ascending=False).head(10).to_string(index=False))
else:
    print('  No gaps detected — this is a single-snapshot file.')
    print('  Gap detection becomes meaningful once multi-hour data is collected.')

## 2.4 QC Summary

In [ ]:
dpc['any_flag'] = dpc['range_flag'] | dpc['flatline_flag']

print('='*55)
print('  STEP 2 QC SUMMARY')
print('='*55)
print(f'  Total observations   : {len(dpc)}')
print(f'  Range violations     : {dpc["range_flag"].sum()}')
print(f'  Flat-line flags      : {dpc["flatline_flag"].sum()}')
print(f'  Any flag             : {dpc["any_flag"].sum()}')
print(f'  Clean observations   : {(~dpc["any_flag"]).sum()}')
print()

# Per-sensor summary
print('  Per-sensor breakdown:')
for c in dpc['sensor_code'].unique():
    sub = dpc[dpc['sensor_code']==c]
    print(f'    {c:8s}: {sub["any_flag"].sum():4d} flagged / {len(sub):5d} total '
          f'({100*sub["any_flag"].sum()/len(sub):.1f}%)')

# Clean dataset for downstream steps
dpc_clean = dpc[~dpc['any_flag']].copy()
print(f'\n  dpc_clean: {len(dpc_clean)} observations')

dpc_clean.to_csv('dpc_clean.csv', index=False)
dpc.to_csv('dpc_flagged.csv', index=False)
print('  Saved -> dpc_clean.csv  |  dpc_flagged.csv')

---
# STEP 3 — Cross-Match DPC vs ERA5

Extracts ERA5 values at DPC station coordinates via bilinear interpolation
for the nearest available ERA5 timestamp, producing a spatial bias profile
across the FVG/Adriatic domain even from a single real-time snapshot.

## 3.1 Load ERA5 for Cross-Match

In [ ]:
def load_era5_month(path, year, month):
    df = pd.read_csv(path)
    sc = [c for c in df.columns if c.startswith('SAMPLE_')]
    base = pd.Timestamp(year=year, month=month, day=1, hour=0, tz='UTC')
    timestamps = [base + pd.Timedelta(hours=i) for i in range(len(sc))]
    samples = df[sc].values * PA_TO_HPA
    valid = ~np.all(np.isnan(samples), axis=0)
    samples = samples[:, valid]
    timestamps = [t for t, v in zip(timestamps, valid) if v]
    return {'timestamps': timestamps, 'samples': samples,
            'lon': df['lon'].values, 'lat': df['lat'].values,
            'lons_uniq': np.sort(df['lon'].unique()),
            'lats_uniq': np.sort(df['lat'].unique())}

era5 = {}
for month_name, meta in ERA5_FILES.items():
    try:
        era5[month_name] = load_era5_month(meta['path'], meta['year'], meta['month'])
        d = era5[month_name]
        print(f'  {month_name:9s}: {d["timestamps"][0]} → {d["timestamps"][-1]}')
    except FileNotFoundError:
        print(f'  {month_name}: NOT FOUND')

if not era5:
    print('No ERA5 files found — Step 3 will be skipped.')

## 3.2 Bilinear Interpolation + Bias Computation

In [ ]:
def bilinear_interp(grid, lats, lons, qlat, qlon):
    qlat = np.clip(qlat, lats[0], lats[-1])
    qlon = np.clip(qlon, lons[0], lons[-1])
    i1 = np.clip(np.searchsorted(lats, qlat)-1, 0, len(lats)-2)
    j1 = np.clip(np.searchsorted(lons, qlon)-1, 0, len(lons)-2)
    i2, j2 = i1+1, j1+1
    lf=(qlat-lats[i1])/(lats[i2]-lats[i1]+1e-10)
    cf=(qlon-lons[j1])/(lons[j2]-lons[j1]+1e-10)
    return float(grid[i1,j1]*(1-lf)*(1-cf)+grid[i2,j1]*lf*(1-cf)
                +grid[i1,j2]*(1-lf)*cf+grid[i2,j2]*lf*cf)

def reshape_grid(vals, lon, lat, lats_u, lons_u):
    g = np.full((len(lats_u), len(lons_u)), np.nan)
    li={v:i for i,v in enumerate(lons_u)}
    la={v:i for i,v in enumerate(lats_u)}
    for v,lo,lt in zip(vals,lon,lat):
        if not (np.isnan(lo) or np.isnan(lt)): g[la[lt],li[lo]]=v
    return g

if not era5:
    print('Skipping Step 3 — no ERA5 files loaded.')
else:
    # Use PRESS sensor from clean DPC data
    dpc_press = dpc_clean[dpc_clean['sensor_code']=='PRESS'].copy()
    dpc_press['dt_utc'] = pd.to_datetime(dpc_press['dt'], utc=True)

    # Build all ERA5 hourly grids into one flat lookup
    all_era5_ts = []
    all_era5_grids = []
    ref_era5 = None
    for month_name, d in era5.items():
        if ref_era5 is None: ref_era5 = d
        for h, ts in enumerate(d['timestamps']):
            all_era5_ts.append(ts)
            all_era5_grids.append(
                reshape_grid(d['samples'][:,h], d['lon'], d['lat'],
                             d['lats_uniq'], d['lons_uniq'])
            )
    era5_times = pd.DatetimeIndex(all_era5_ts)

    rows = []
    for _, row in dpc_press.iterrows():
        deltas = np.abs(era5_times - row['dt_utc'])
        mi = deltas.argmin()
        era5_val = bilinear_interp(
            all_era5_grids[mi],
            ref_era5['lats_uniq'], ref_era5['lons_uniq'],
            row['lat'], row['lon']
        )
        rows.append({'station_id':row['station_id'],
                     'lat':row['lat'],'lon':row['lon'],
                     'dt':row['dt_utc'],'era5_dt':all_era5_ts[mi],
                     'dpc_press':row['value'],'era5_press':era5_val,
                     'bias':row['value']-era5_val})

    xmatch = pd.DataFrame(rows)
    print(f'Cross-match rows   : {len(xmatch)}')
    print(f'Mean bias (hPa)    : {xmatch["bias"].mean():+.3f}')
    print(f'Std bias (hPa)     : {xmatch["bias"].std():.3f}')
    xmatch.to_csv('dpc_era5_crossmatch.csv', index=False)
    print('Saved -> dpc_era5_crossmatch.csv')

## 3.3 Spatial Bias Map

In [ ]:
if 'xmatch' in dir() and len(xmatch)>0:
    vmax = max(abs(xmatch['bias'].min()), abs(xmatch['bias'].max()))
    norm = mcolors.TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

    sc1 = ax1.scatter(xmatch['lon'], xmatch['lat'],
                      c=xmatch['bias'], cmap='RdBu_r', norm=norm,
                      s=80, edgecolors='black', linewidths=0.4, zorder=5)
    plt.colorbar(sc1, ax=ax1, label='Bias: DPC − ERA5 (hPa)')
    ax1.set(xlim=(LON_MIN-0.3, LON_MAX+0.3), ylim=(LAT_MIN-0.3, LAT_MAX+0.3),
            xlabel='Lon (°E)', ylabel='Lat (°N)',
            title='Step 3 — DPC vs ERA5 Bias Map\n(PRESS)')

    ax2.scatter(xmatch['era5_press'], xmatch['dpc_press'],
                alpha=0.5, s=20, color='steelblue')
    lims=[min(xmatch['era5_press'].min(),xmatch['dpc_press'].min())-1,
          max(xmatch['era5_press'].max(),xmatch['dpc_press'].max())+1]
    ax2.plot(lims,lims,'k--',linewidth=1.2,label='1:1')
    bias_m = xmatch['bias'].mean()
    rmse   = np.sqrt((xmatch['bias']**2).mean())
    r      = xmatch[['era5_press','dpc_press']].corr().iloc[0,1]
    ax2.set(xlabel='ERA5 PRESS (hPa)', ylabel='DPC PRESS (hPa)',
            title=f'Step 3 — Scatter\nbias={bias_m:+.2f}  RMSE={rmse:.2f}  r={r:.3f}')
    ax2.legend(fontsize=9)

    fig.suptitle('Step 3 — DPC vs ERA5 Cross-Match (PRESS)',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('s3_dpc_era5_bias.png', bbox_inches='tight')
    plt.show()
else:
    print('Step 3 skipped — no ERA5 data or no cross-match results.')

---
# STEP 4 — Radar QC (GeoTIFF)

Quality control for DPC Radar raster products (SRI, VMI, HRD):
- **Coverage check** — fraction of valid pixels per file
- **Artefact detection** — range-ring and radial spike detection
- **Temporal alignment** — verify radar timestamps align with station obs

> **Note:** This step requires GeoTIFF radar files in `RADAR_DIR`.
> If no radar files are available, the cells will report this and skip gracefully.

## 4.1 Radar File Inventory

In [ ]:
radar_files = sorted(glob.glob(os.path.join(RADAR_DIR, '**/*.tif'), recursive=True)) + \
              sorted(glob.glob(os.path.join(RADAR_DIR, '**/*.tiff'), recursive=True))

print(f'Radar GeoTIFF files found: {len(radar_files)}')

if not radar_files:
    print(f'  No radar files in {RADAR_DIR}')
    print('  Step 4 requires DPC Radar GeoTIFFs.')
    print('  Download with: https://github.com/pcm-dpc/DPC-Radar-data-downloader')
    RADAR_AVAILABLE = False
else:
    RADAR_AVAILABLE = True
    # Parse product code and timestamp from filenames
    import re
    records = []
    for f in radar_files:
        fname = os.path.basename(f)
        size_mb = os.path.getsize(f) / 1e6
        # Try to extract timestamp from filename (YYYYMMDDHHMMSS pattern)
        ts_match = re.search(r'(\d{14})', fname)
        ts = pd.to_datetime(ts_match.group(1), format='%Y%m%d%H%M%S') if ts_match else None
        # Extract product code (SRI, VMI, HRD, etc.)
        prod_match = re.search(r'(SRI|VMI|HRD|SRT|LTG|TMP|CAPPI)', fname, re.IGNORECASE)
        product = prod_match.group(1).upper() if prod_match else 'UNKNOWN'
        records.append({'file': fname, 'product': product,
                        'timestamp': ts, 'size_mb': round(size_mb, 2)})
    radar_inv = pd.DataFrame(records)
    print(radar_inv.groupby('product')[['file']].count().rename(columns={'file':'n_files'}).to_string())
    print()
    print(radar_inv.head(10).to_string(index=False))

## 4.2 Coverage Check — Valid Pixel Fraction

In [ ]:
if not RADAR_AVAILABLE:
    print('Skipping — no radar files available.')
else:
    try:
        import rasterio
        RASTERIO_OK = True
    except ImportError:
        print('rasterio not installed. Run: pip install rasterio')
        RASTERIO_OK = False

    if RASTERIO_OK:
        coverage_results = []
        for f in radar_files[:20]:  # check first 20 files
            try:
                with rasterio.open(f) as src:
                    data = src.read(1).astype(float)
                    nodata = src.nodata
                    if nodata is not None:
                        valid = data != nodata
                    else:
                        valid = ~np.isnan(data) & (data > -9999)
                    pct_valid = 100 * valid.sum() / data.size
                    pct_zero  = 100 * (data[valid]==0).sum() / valid.sum() if valid.sum()>0 else 0
                    coverage_results.append({
                        'file': os.path.basename(f),
                        'shape': f'{data.shape[0]}x{data.shape[1]}',
                        'pct_valid': round(pct_valid,1),
                        'pct_zero_of_valid': round(pct_zero,1),
                        'min': round(float(data[valid].min()),3) if valid.sum()>0 else np.nan,
                        'max': round(float(data[valid].max()),3) if valid.sum()>0 else np.nan,
                    })
            except Exception as e:
                print(f'  Error reading {f}: {e}')

        cov_df = pd.DataFrame(coverage_results)
        print('Coverage summary:')
        print(cov_df.to_string(index=False))

        # Flag files with poor coverage
        low_cov = cov_df[cov_df['pct_valid'] < 50]
        print(f'\nFiles with < 50% valid pixels: {len(low_cov)}')

## 4.3 Artefact Detection — Range-Ring and Radial Spikes

In [ ]:
if not RADAR_AVAILABLE:
    print('Skipping — no radar files available.')
elif not RASTERIO_OK:
    print('Skipping — rasterio not installed.')
else:
    def detect_radial_spikes(data, threshold_ratio=5.0):
        """
        Detect radial spike artefacts: columns/rows where mean is
        threshold_ratio times higher than the global mean.
        Returns fraction of flagged rows and columns.
        """
        valid = data[~np.isnan(data) & (data > 0)]
        if len(valid) == 0: return 0.0, 0.0
        global_mean = valid.mean()
        row_means = np.nanmean(data, axis=1)
        col_means = np.nanmean(data, axis=0)
        spiked_rows = (row_means > threshold_ratio * global_mean).sum()
        spiked_cols = (col_means > threshold_ratio * global_mean).sum()
        return spiked_rows / data.shape[0], spiked_cols / data.shape[1]

    artefact_results = []
    for f in radar_files[:20]:
        try:
            with rasterio.open(f) as src:
                data = src.read(1).astype(float)
                nodata = src.nodata
                if nodata is not None: data[data==nodata] = np.nan
                frac_row, frac_col = detect_radial_spikes(data)
                artefact_results.append({
                    'file': os.path.basename(f),
                    'spike_row_frac': round(frac_row,4),
                    'spike_col_frac': round(frac_col,4),
                    'flagged': (frac_row > 0.05 or frac_col > 0.05)
                })
        except Exception as e:
            print(f'  Error: {f}: {e}')

    art_df = pd.DataFrame(artefact_results)
    flagged = art_df[art_df['flagged']]
    print(f'Files with radial spike artefacts: {len(flagged)} / {len(art_df)}')
    if len(flagged)>0:
        print(flagged.to_string(index=False))

## 4.4 Temporal Alignment — Radar vs Station

In [ ]:
if not RADAR_AVAILABLE:
    print('Skipping — no radar files available.')
else:
    # Check that radar timestamps overlap with DPC station observation window
    if 'radar_inv' in dir() and radar_inv['timestamp'].notna().sum()>0:
        radar_times = radar_inv['timestamp'].dropna().sort_values()
        station_times = dpc['dt'].sort_values()

        print('Radar time range  :', radar_times.min(), '→', radar_times.max())
        print('Station time range:', station_times.min(), '→', station_times.max())

        # Check overlap
        radar_min, radar_max = radar_times.min(), radar_times.max()
        sta_min, sta_max = station_times.min(), station_times.max()
        overlap_start = max(radar_min, sta_min)
        overlap_end   = min(radar_max, sta_max)

        if overlap_start < overlap_end:
            print(f'\nTemporal overlap  : {overlap_start} → {overlap_end}')
            overlap_radar = radar_inv[
                (radar_inv['timestamp']>=overlap_start) &
                (radar_inv['timestamp']<=overlap_end)
            ]
            print(f'Radar files in overlap window: {len(overlap_radar)}')
        else:
            print('\nNo temporal overlap between radar and station data.')
            print('Check that radar files and station CSVs cover the same period.')
    else:
        print('Radar timestamps not parseable from filenames.')
        print('Verify filename format contains YYYYMMDDHHMMSS.')

---
# Final Summary

In [ ]:
print('='*60)
print('  DPC DATA QUALITY PIPELINE — SUMMARY')
print('='*60)
print(f'  STEP 1 — Inventory')
print(f'    Total stations     : {dpc["station_id"].nunique()}')
print(f'    Total sensors      : {dpc["sensor_code"].nunique()}')
print(f'    Sensor-station pairs: {inventory["station_id"].nunique() * dpc["sensor_code"].nunique()}')
print()
print(f'  STEP 2 — QC')
print(f'    Range violations   : {dpc["range_flag"].sum()}')
print(f'    Flat-line flags    : {dpc["flatline_flag"].sum()}')
print(f'    Clean observations : {len(dpc_clean)}')
print()
if 'xmatch' in dir() and len(xmatch)>0:
    print(f'  STEP 3 — ERA5 Cross-match')
    print(f'    Matched pairs      : {len(xmatch)}')
    print(f'    Mean bias (hPa)    : {xmatch["bias"].mean():+.3f}')
    print(f'    Correlation r      : {xmatch[["era5_press","dpc_press"]].corr().iloc[0,1]:.3f}')
else:
    print(f'  STEP 3 — Skipped (no ERA5 files)')
print()
if RADAR_AVAILABLE:
    print(f'  STEP 4 — Radar QC')
    print(f'    Radar files found  : {len(radar_files)}')
else:
    print(f'  STEP 4 — Skipped (no radar files)')
print()
print('  Saved files:')
for f in ['dpc_data_dictionary.csv','dpc_clean.csv','dpc_flagged.csv',
          'dpc_era5_crossmatch.csv',
          's1_sensor_availability.png','s1_station_map.png',
          's2_range_check.png','s3_dpc_era5_bias.png']:
    print(f'    {f}')
print('='*60)